# 05 · CI/CD with Jobs as Code

**Goal:** build an automated validation pipeline in code, and watch its **quality gate** block a model that is not good enough.

**CI/CD in one picture:** every change runs through the same automatic checks before it can reach production.

```
 change ──► validate data ──► train model ──► quality gate ──► register (release)
                                                  │
                                          AUC too low? ──► stop: nothing is released
```

In the full course the trigger is a **Git commit** and the job is created with the **Databricks CLI**. Here the job is
created with the **Databricks SDK** (Python), which sends the same job definition to the same Jobs API.

In [ ]:
%run ./Includes/_config

In [ ]:
import base64, time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
TASK_DIR = f"{here.rsplit('/', 1)[0]}/05_pipeline_tasks"
CI_MODEL = f"{CATALOG}.{SCHEMA}.diabetes_ci_model"
JOB_NAME = f"SWSAcademy - Concepts CI Pipeline ({slug})"
print("Task notebooks will be written to:", TASK_DIR)

## 1. Write the four task notebooks
Each task is a small notebook. Tasks pass results to each other with **task values**
(`dbutils.jobs.taskValues.set` and `.get`).

In [ ]:
TASKS = {
"validate_data": f'''
table = dbutils.widgets.get("table")
df = spark.table(table)
rows = df.count()
bad_labels = df.filter("Diabetes_binary NOT IN (0, 1)").count()
print(f"rows={{rows}} bad_labels={{bad_labels}}")
assert rows > 10000 and bad_labels == 0, "Data validation failed"
''',
"train_model": f'''
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
mlflow.set_experiment("{EXPERIMENT}")
pdf = spark.table(dbutils.widgets.get("table")).sample(fraction=0.3, seed=7).toPandas()
features = [c for c in pdf.columns if c not in ("Diabetes_binary", "id")]
X_tr, X_te, y_tr, y_te = train_test_split(pdf[features], pdf["Diabetes_binary"], test_size=0.25, random_state=7)
with mlflow.start_run(run_name="ci_pipeline_train") as run:
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    mlflow.log_metric("test_auc", auc)
    mlflow.sklearn.log_model(model, "model", input_example=X_tr.head(3))
print(f"AUC = {{auc:.4f}}")
dbutils.jobs.taskValues.set(key="auc", value=float(auc))
dbutils.jobs.taskValues.set(key="run_id", value=run.info.run_id)
''',
"quality_gate": '''
auc = float(dbutils.jobs.taskValues.get(taskKey="train_model", key="auc", debugValue=0.0))
min_auc = float(dbutils.widgets.get("min_auc"))
print(f"AUC {auc:.4f} vs required {min_auc}")
if auc < min_auc:
    raise Exception(f"Quality gate FAILED: AUC {auc:.4f} is below {min_auc}. Release blocked.")
print("Quality gate PASSED")
''',
"register_model": f'''
import mlflow
mlflow.set_registry_uri("databricks-uc")
run_id = dbutils.jobs.taskValues.get(taskKey="train_model", key="run_id", debugValue="")
mv = mlflow.register_model(f"runs:/{{run_id}}/model", "{CI_MODEL}")
print(f"Released {CI_MODEL} version {{mv.version}}")
''',
}

w.workspace.mkdirs(TASK_DIR)
for name, code in TASKS.items():
    source = "# Databricks notebook source\n" + code.strip() + "\n"
    w.api_client.do("POST", "/api/2.0/workspace/import", body={
        "path": f"{TASK_DIR}/{name}", "format": "SOURCE", "language": "PYTHON", "overwrite": True,
        "content": base64.b64encode(source.encode()).decode()})
print("Wrote:", list(TASKS))

## 2. Define the job as code
The job is plain data: a list of tasks with dependencies, and job-level **parameters** every task can read.
No compute is specified, so every task runs on serverless.

In [ ]:
def task(key, depends_on=None):
    t = {"task_key": key, "notebook_task": {"notebook_path": f"{TASK_DIR}/{key}", "source": "WORKSPACE"}}
    if depends_on:
        t["depends_on"] = [{"task_key": depends_on}]
    return t

job_settings = {
    "name": JOB_NAME,
    "tags": {"course": "swsacademy", "purpose": "concepts-ci"},
    "parameters": [{"name": "table", "default": TABLE}, {"name": "min_auc", "default": "0.75"}],
    "tasks": [task("validate_data"), task("train_model", "validate_data"),
              task("quality_gate", "train_model"), task("register_model", "quality_gate")],
}

existing = [j for j in w.jobs.list(name=JOB_NAME)]
if existing:
    job_id = existing[0].job_id
    w.api_client.do("POST", "/api/2.1/jobs/reset", body={"job_id": job_id, "new_settings": job_settings})
else:
    job_id = w.api_client.do("POST", "/api/2.1/jobs/create", body=job_settings)["job_id"]
print(f"Job '{JOB_NAME}' id={job_id}")

def run_and_report(min_auc):
    run_id = w.api_client.do("POST", "/api/2.1/jobs/run-now",
                             body={"job_id": job_id, "job_parameters": {"min_auc": str(min_auc)}})["run_id"]
    while True:
        run = w.api_client.do("GET", "/api/2.1/jobs/runs/get", query={"run_id": run_id})
        if run["state"]["life_cycle_state"] in ("TERMINATED", "INTERNAL_ERROR", "SKIPPED"):
            break
        time.sleep(15)
    print(f"min_auc={min_auc}: job {run['state'].get('result_state')}")
    for t in sorted(run["tasks"], key=lambda t: t.get("start_time", 0) or 9e15):
        print(f"   {t['task_key']:15} {t['state'].get('result_state', t['state']['life_cycle_state'])}")
    return run

## 3. Run 1: an impossible bar (AUC ≥ 0.99)
The gate fails, so `register_model` never runs. **Nothing is released.** This is CI doing its job.

In [ ]:
failed_run = run_and_report(min_auc=0.99)

## 4. Run 2: a realistic bar (AUC ≥ 0.75)
Every task succeeds and a new model version is registered in Unity Catalog.

In [ ]:
passed_run = run_and_report(min_auc=0.75)
assert passed_run["state"].get("result_state") == "SUCCESS", "The pipeline should pass with min_auc = 0.75"
print("\nOpen Jobs & Pipelines >", JOB_NAME, "> Runs to see both runs side by side.")

## Key points
- A job definition is **data** (JSON or YAML). Keeping it in Git makes the pipeline reviewable and repeatable.
- **Task values** pass results (here, the AUC) from one task to the next.
- A **quality gate** turns "the model looks fine" into an automatic yes/no that blocks bad releases.
- The full course triggers this on every Git commit with the Databricks CLI; notebook 08 shows the same job as an Asset Bundle.